
# 🌸 Project 22: My Little Journal
## Notebook 02 — Data Preprocessing & Feature Engineering

**Objective:** Prepare the GoEmotions dataset for multi-label emotion classification by addressing data leakage, processing text, and encoding emotion labels.

**Dataset:** GoEmotions (Google Research)



### 1. Load Raw Dataset

Load the original training, validation, and test datasets for preprocessing.


In [11]:

import pandas as pd
import numpy as np
from pathlib import Path

# Define project directories
project_dir = Path.cwd().parent
data_dir = project_dir / "data" / "raw"

# Load original dataset splits
train_df = pd.read_csv(data_dir / "train.csv")
val_df = pd.read_csv(data_dir / "validation.csv")
test_df = pd.read_csv(data_dir / "test.csv")

# Display dataset sizes
print("Training:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)


Training: (43410, 3)
Validation: (5426, 3)
Test: (5427, 3)



### 2. Remove Cross-Split Text Overlap

Remove overlapping text entries to prevent potential data leakage while preserving the original test set.


In [12]:

# Record original dataset sizes
original_train = len(train_df)
original_val = len(val_df)

# Keep the test set unchanged
test_texts = set(test_df["text"])

# Remove training rows appearing in validation or test
excluded_train_texts = set(val_df["text"]) | test_texts

train_df = train_df[
    ~train_df["text"].isin(excluded_train_texts)
].copy()

# Remove validation rows appearing in test
val_df = val_df[
    ~val_df["text"].isin(test_texts)
].copy()

# Reset DataFrame indices
train_df.reset_index(drop=True, inplace=True)
val_df.reset_index(drop=True, inplace=True)

# Display cleaning results
print("Training rows removed:", original_train - len(train_df))
print("Validation rows removed:", original_val - len(val_df))

print("\nCleaned dataset sizes:")
print("Training:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)


Training rows removed: 126
Validation rows removed: 10

Cleaned dataset sizes:
Training: (43284, 3)
Validation: (5416, 3)
Test: (5427, 3)



### 3. Verify Cross-Split Overlap Removal

Confirm that no identical text entries remain across the cleaned training, validation, and test datasets.


In [13]:

# Convert text columns to sets
train_texts = set(train_df["text"])
val_texts = set(val_df["text"])
test_texts = set(test_df["text"])

# Verify overlaps
print("Train–Validation overlap:", len(train_texts & val_texts))
print("Train–Test overlap:", len(train_texts & test_texts))
print("Validation–Test overlap:", len(val_texts & test_texts))

# Assert that no exact cross-split overlaps remain
assert train_texts.isdisjoint(val_texts)
assert train_texts.isdisjoint(test_texts)
assert val_texts.isdisjoint(test_texts)

print("\n✅ No exact cross-split text overlap detected!")


Train–Validation overlap: 0
Train–Test overlap: 0
Validation–Test overlap: 0

✅ No exact cross-split text overlap detected!



### 4. Convert Emotion Labels to Lists

Convert emotion labels from CSV strings into lists of integers for multi-label encoding.


In [14]:

# Convert string labels into lists of integers
for split_df in [train_df, val_df, test_df]:
    split_df["labels"] = split_df["labels"].apply(
        lambda x: np.fromstring(x.strip("[]"), sep=" ", dtype=int).tolist()
    )

# Verify conversion
print("Example labels:", train_df["labels"].iloc[7])
print("Data type:", type(train_df["labels"].iloc[7]))

train_df[["text", "labels"]].head(10)


Example labels: [8, 20]
Data type: <class 'list'>


,text,labels
0,My favourite food is anything I didn't have to...,[27]
1,"Now if he does off himself, everyone will thin...",[27]
2,WHY THE FUCK IS BAYLESS ISOING,[2]
3,To make her feel threatened,[14]
4,Dirty Southern Wankers,[3]
5,OmG pEyToN iSn'T gOoD eNoUgH tO hElP uS iN tHe...,[26]
6,Yes I heard abt the f bombs! That has to be wh...,[15]
7,We need more boards and to create a bit more s...,"[8, 20]"
8,Damn youtube and outrage drama is super lucrat...,[0]
9,It might be linked to the trust factor of your...,[27]



### 5. Validate Emotion Labels

Verify that each example contains valid emotion IDs between 0 and 27.


In [15]:

# GoEmotions contains 28 labels (0–27)
num_labels = 28

for split_name, split_df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    invalid_rows = split_df["labels"].apply(
        lambda labels: (
            len(labels) == 0
            or any(label < 0 or label >= num_labels for label in labels)
        )
    )

    print(f"{split_name} invalid rows: {invalid_rows.sum()}")

    assert invalid_rows.sum() == 0, (
        f"Invalid emotion labels found in {split_name}!"
    )

print("\nAll emotion labels are valid!")


Train invalid rows: 0
Validation invalid rows: 0
Test invalid rows: 0

All emotion labels are valid!



### 6. Multi-Hot Label Encoding

Convert lists of emotion IDs into 28-dimensional binary vectors for multi-label classification.


In [16]:

from sklearn.preprocessing import MultiLabelBinarizer

# Create the encoder with all 28 emotion classes
mlb = MultiLabelBinarizer(classes=range(28))

# Fit on the predefined label classes
mlb.fit([[]])

# Transform labels for each dataset split
y_train = mlb.transform(train_df["labels"])
y_val = mlb.transform(val_df["labels"])
y_test = mlb.transform(test_df["labels"])

# Check the results
print("Training labels:", y_train.shape)
print("Validation labels:", y_val.shape)
print("Test labels:", y_test.shape)

print("\nOriginal labels:", train_df["labels"].iloc[0])
print("Encoded labels:", y_train[0])


Training labels: (43284, 28)
Validation labels: (5416, 28)
Test labels: (5427, 28)

Original labels: [27]
Encoded labels: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1]



### 7. Prepare Text Features

Extract text inputs for model training while keeping the corresponding multi-hot encoded labels.


In [17]:

# Extract text features
X_train = train_df["text"].astype(str)
X_val = val_df["text"].astype(str)
X_test = test_df["text"].astype(str)

# Check dataset dimensions
print("Training texts:", X_train.shape)
print("Validation texts:", X_val.shape)
print("Test texts:", X_test.shape)

# Display one example
print("\nExample text:", X_train.iloc[0])
print("Emotion IDs:", train_df["labels"].iloc[0])
print("Multi-hot vector:", y_train[0])


Training texts: (43284,)
Validation texts: (5416,)
Test texts: (5427,)

Example text: My favourite food is anything I didn't have to cook myself.
Emotion IDs: [27]
Multi-hot vector: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1]



### 8. Save Processed Dataset

Save the cleaned dataset splits and multi-hot encoded labels for reproducible model training.


In [18]:

# Create processed data directory
processed_dir = project_dir / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

# Save cleaned datasets
train_df.to_pickle(processed_dir / "train.pkl")
val_df.to_pickle(processed_dir / "validation.pkl")
test_df.to_pickle(processed_dir / "test.pkl")

# Save multi-hot encoded labels
np.save(processed_dir / "y_train.npy", y_train)
np.save(processed_dir / "y_val.npy", y_val)
np.save(processed_dir / "y_test.npy", y_test)

print("Processed datasets saved successfully!")
print("Location:", processed_dir.resolve())


Processed datasets saved successfully!
Location: C:\Projects\22_My_Little_Journal_app\data\processed



### 9. Verify Processed Files

Confirm that all processed datasets and encoded labels have been saved successfully.


In [19]:

# Verify saved files
expected_files = [
    "train.pkl",
    "validation.pkl",
    "test.pkl",
    "y_train.npy",
    "y_val.npy",
    "y_test.npy"
]

for filename in expected_files:
    file_path = processed_dir / filename
    print(f"{filename}: {'Found' if file_path.is_file() else 'Missing'}")

assert all(
    (processed_dir / filename).is_file()
    for filename in expected_files
)

print("\nAll processed files verified successfully!")


train.pkl: Found
validation.pkl: Found
test.pkl: Found
y_train.npy: Found
y_val.npy: Found
y_test.npy: Found

All processed files verified successfully!



## Conclusion

The GoEmotions dataset was successfully preprocessed and prepared for multi-label emotion classification.

### Key Achievements
- Removed exact text overlaps across dataset splits to reduce potential data leakage.
- Preserved the original test set.
- Converted emotion labels into lists of integers.
- Validated all 28 emotion IDs.
- Applied multi-hot encoding to support multiple emotions per text.
- Prepared text inputs (X) and target labels (y).
- Saved the cleaned datasets and encoded labels for model training.

### Final Dataset Sizes
- Training: 43,284 examples
- Validation: 5,416 examples
- Test: 5,427 examples

### Next Steps
- Extract numerical text features using TF-IDF.
- Train a baseline multi-label classification model.
- Evaluate performance using precision, recall, and F1-score.
